# NCTBench-Pilot — Steps 3 and 4: Chunking and Embedding
RecursiveCharacterTextSplitter (1000 chars, 200 overlap).
Bengali danda (।) as primary separator.
intfloat/multilingual-e5-large with passage: prefix.

In [ ]:
import sys, json, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
from config import (CHUNKS_DIR, EMBEDDINGS_DIR,
                    CHROMA_COLLECTION, EMBEDDING_MODEL, QUERY_PREFIX)

## Step 3: Chunking

In [ ]:
from step3_chunk import run_chunking
run_chunking()

In [ ]:
chunks = json.loads(
    (CHUNKS_DIR / "all_chunks.json").read_text(encoding="utf-8")
)
print(f"Total chunks: {len(chunks)}")
print("By language :", dict(Counter(c["language"] for c in chunks)))
print("By subject  :", dict(Counter(c["subject_code"] for c in chunks)))
print("By class    :", dict(sorted(
    Counter(c["class_num"] for c in chunks).items()
)))
print("By grade    :", dict(Counter(c["grade_band"] for c in chunks)))

In [ ]:
bn = next(c for c in chunks if c["language"] == "bn")
en = next(c for c in chunks if c["language"] == "en")
print("=== BENGALI CHUNK ===")
for k, v in bn.items():
    if k != "text":
        print(f"  {k:20s}: {v}")
print(f"  {'text':20s}: {bn['text'][:300]}...")
print("\n=== ENGLISH CHUNK ===")
for k, v in en.items():
    if k != "text":
        print(f"  {k:20s}: {v}")
print(f"  {'text':20s}: {en['text'][:300]}...")

In [ ]:
lengths = [c["char_count"] for c in chunks]
bn_l = [c["char_count"] for c in chunks if c["language"] == "bn"]
en_l = [c["char_count"] for c in chunks if c["language"] == "en"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(bn_l, bins=40, color="#0072B2", edgecolor="white")
axes[0].set_title("Bengali Chunk Lengths")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")
axes[1].hist(en_l, bins=40, color="#E69F00", edgecolor="white")
axes[1].set_title("English Chunk Lengths")
axes[1].set_xlabel("Characters")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
plt.suptitle("Chunk Length Distribution", fontsize=13)
plt.tight_layout()
plt.savefig(r"E:\CSAA Project\notebooks\chunk_lengths.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean: {sum(lengths)/len(lengths):.0f}  "
      f"Min: {min(lengths)}  Max: {max(lengths)}")

## Step 4: Embedding
The passage: prefix is REQUIRED by multilingual-e5-large.
Omitting it produces incorrect cosine similarity scores.

In [ ]:
from step4_embed_index import run_embedding
run_embedding()

In [ ]:
import chromadb
client = chromadb.PersistentClient(path=str(EMBEDDINGS_DIR))
col = client.get_collection(CHROMA_COLLECTION)
print(f"Collection : {CHROMA_COLLECTION}")
print(f"Records    : {col.count()}")

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMBEDDING_MODEL)
query = "What is the function of RAM in a computer?"
q_vec = model.encode(
    f"{QUERY_PREFIX}{query}",
    normalize_embeddings=True
).tolist()
results = col.query(
    query_embeddings=[q_vec], n_results=3,
    include=["metadatas"]
)
print(f"Query: {query}\n")
for i, meta in enumerate(results["metadatas"][0]):
    print(f"Result {i+1}: {meta['source_pdf']} | "
          f"class{meta['class_num']} | {meta['language']}")
    print(f"  {meta['text_preview'][:200]}")
    print()